In [1]:
from _bootstrap import setup

setup()

PosixPath('/home/arieltr/Projects/master-thesis/source/ai-agent')

In [ ]:
import numpy as np
import open3d as o3d

from geometry.curvature import cluster_cavities

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
mesh = o3d.io.read_triangle_mesh("/home/arieltr/Desktop/MODELO_A/SN_EST_1_A_high.obj")

mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh.remove_duplicated_vertices()
mesh.remove_non_manifold_edges()

nverts = len(mesh.vertices)
nfaces = len(mesh.triangles)

print(f"Mesh has {nverts} vertices and {nfaces} faces.")

[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveNonManifoldEdges] This mesh contains triangle uvs that are not handled in this function
Mesh has 1858610 vertices and 3716109 faces.


In [4]:
extent = np.linalg.norm(mesh.get_axis_aligned_bounding_box().get_extent())

mesh = mesh.simplify_vertex_clustering(
    voxel_size=extent / 500.0
)

nverts = len(mesh.vertices)
nfaces = len(mesh.triangles)

print(f"Simplified mesh has {nverts} vertices and {nfaces} faces.")

Simplified mesh has 288700 vertices and 590790 faces.


In [ ]:
clusters = cluster_cavities(mesh, percentile=10.0, min_points=5)

dent_vertices = sum(len(cluster) for cluster in clusters)
print(f"Found {len(clusters)} distinct dent areas ({dent_vertices} clustered vertices).")

Found 1433 distinct dent areas (16683 clustered vertices).


In [6]:
vertices = np.asarray(mesh.vertices)
colors = np.ones_like(vertices) * 0.5

if clusters:
    o3d_vertices = o3d.utility.Vector3dVector(vertices)

    for cluster_indices in clusters:
        cluster_points = vertices[cluster_indices]
        min_bound = np.min(cluster_points, axis=0)
        max_bound = np.max(cluster_points, axis=0)
        bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
        in_bounds_indices = bbox.get_point_indices_within_bounding_box(o3d_vertices)
        colors[in_bounds_indices] = [1.0, 0.0, 0.0]
else:
    print("No dent clusters found.")

In [7]:
mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([mesh])